In [2]:
import torch
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
import numpy as np

# 1. Load the fine-tuned model and processor from your local directory
model_path = "./final_model"
processor = SegformerImageProcessor.from_pretrained(model_path)
model = SegformerForSemanticSegmentation.from_pretrained(model_path)

# Move model to CPU for a cleaner export
model.to("cpu")
model.eval()

# CRITICAL: Tell Hugging Face to return standard tuples instead of dictionaries
model.config.return_dict = False 

# 2. Prepare a dummy input for ONNX tracing
# Create a dummy image (e.g., 512x512 RGB) matching your training data resolution
dummy_image = np.zeros((512, 512, 3), dtype=np.uint8)

# Run it through the processor to get the correctly normalized 'pixel_values' tensor
inputs = processor(images=dummy_image, return_tensors="pt")
dummy_pixel_values = inputs["pixel_values"]

# 3. Export to ONNX
onnx_file_path = "segformer.onnx"

torch.onnx.export(
    model,
    (dummy_pixel_values,),             # Must be wrapped in a tuple
    onnx_file_path,
    export_params=True,
    opset_version=14,                  # Opset 14 is highly stable for Vision Transformers
    input_names=['pixel_values'],      # Segformer explicitly expects this input name
    output_names=['logits'],           # The output is the unnormalized predictions
    dynamic_axes={
        'pixel_values': {0: 'batch_size', 2: 'height', 3: 'width'},
        'logits': {0: 'batch_size', 2: 'height', 3: 'width'}
    }
)

print(f"Model successfully exported to {onnx_file_path}")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Model successfully exported to segformer.onnx
